In [ ]:
# import os, sys
# project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
# if project_root not in sys.path:
#     sys.path.append(project_root)
# from agents.hotel.agent import hotel_graph as graph
import os, sys
import uuid
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Import hàm build graph thay vì import graph trực tiếp
# from agents.hotel.agent import hotel_graph as graph 
from agents.hotel.agent import build_hotel_graph 
from utils.tracing import with_trace_config

# Khởi tạo graph (cần dùng await vì hàm này là async)
graph = await build_hotel_graph()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    pass

In [ ]:
# def stream_graph_updates(inputs: dict):
#     for event in graph.stream(inputs):
#         if "hotel_chat" in event:
#             messages = event["hotel_chat"].get("messages", [])
#             for msg in messages:
#                 # Nếu là AIMessage (object)
#                 if hasattr(msg, "content") and msg.content:
#                     print("Assistant:", msg.content)
#                 # Nếu là dict
#                 elif isinstance(msg, dict) and msg.get("role") == "assistant":
#                     print("Assistant:", msg.get("content"))
#         if "tools" in event:
#             messages = event["tools"].get("messages", [])
#             for msg in messages:
#                 if hasattr(msg, "content") and msg.content:
#                     print("Tool Output:", msg.content)
#                 elif isinstance(msg, dict):
#                     print("Tool Output:", msg.get("content"))
# async def stream_graph_updates(inputs: dict):
#     async for event in graph.astream(inputs):
#         if "hotel_chat" in event:
#             messages = event["hotel_chat"].get("messages", [])
#             for msg in messages:
#                 if hasattr(msg, "content") and msg.content:
#                     print("Assistant:", msg.content)
#                 elif isinstance(msg, dict) and msg.get("role") == "assistant":
#                     print("Assistant:", msg.get("content"))

#         if "tools" in event:
#             messages = event["tools"].get("messages", [])
#             for msg in messages:
#                 if hasattr(msg, "content") and msg.content:
#                     print("Tool Output:", msg.content)
#                 elif isinstance(msg, dict):
#                     print("Tool Output:", msg.get("content"))
from langchain_core.messages import AIMessage

async def stream_graph_updates(inputs: dict):
    thread_id = str(uuid.uuid4())
    config = with_trace_config(
        {"configurable": {"thread_id": thread_id}},
        run_name="hotel_agent_notebook",
        tags=["customer-support", "hotel", "notebook"],
        metadata={"thread_id": thread_id, "agent": "hotel"},
    )

    async for event in graph.astream(inputs, config=config):
        if "hotel_chat" in event:
            for msg in event["hotel_chat"].get("messages", []):
                # In tham số tool — đây là chỗ cần check
                if isinstance(msg, AIMessage) and msg.tool_calls:
                    for tc in msg.tool_calls:
                        print("🔧 Tool:", tc["name"])
                        print("📥 Args:", tc["args"])

                if hasattr(msg, "content") and msg.content:
                    print("Assistant:", msg.content)

        if "tools" in event:
            for msg in event["tools"].get("messages", []):
                if hasattr(msg, "content") and msg.content:
                    print("Tool Output:", msg.content)

In [ ]:
test_cases = [
    "Khách sạn nào mà cách trung tâm gần á. Tìm kiếm khách sạn ở Hà Nội,  giá thấp nhất là 2 triệu, rating trên 9.0, cho 4 người lớn, check in 20/9/2026 và check out 25/9/2026.",
]

for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs)


In [ ]:
test_cases = [
    "Tìm kiếm khách sạn ở Đà Nẵng, 2 người lớn, có 1 trẻ em 16 tuổi, check in 20/8/2026, check out 22/8/2026. ",
]

for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs)

In [ ]:
test_cases = [
    "Tìm kiếm khách sạn ở Phú Yên, giá là 2 triệu VND, đánh giá trên 8, 2 người lớn, ccheck in 20/9/2026, check out 22/9/2026. ",
]

for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs)

In [ ]:
test_cases = [
    "Tìm kiếm khách sạn ở Nha Trang, giá cao nhất là 2 triệu VND, đánh giá trên 9, 2 người lớn, có 1 trẻ em 5 tuổi, ccheck in 20/9/2026, check out 22/9/2026. ",
]

for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs)

In [ ]:
test_cases = [
    "Tìm kiếm khách sạn ở Huế, giá hạng sang, đánh giá trên 8, 2 người lớn, có 1 trẻ em 5 tuổi, ccheck in 20/9/2026, check out 22/9/2026. ",
]

for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs)

In [ ]:
test_cases = [
    "Tôi muốn xem phòng ở khách sạn có ID là 266027, check in 27/7/2026, check out 29/7/2026, cho 2 người lớn.",
]

for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs)

In [ ]:
test_cases = [
    "Tìm kiếm khách sạn ở Huế, giá rẻ, đánh giá trên 8, 2 người lớn, có 1 trẻ em 5 tuổi, ccheck in 20/9/2026, check out 22/9/2026. ",
]

for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs)



In [ ]:
test_cases = [
    "Tôi muốn xem phòng ở khách sạn có ID là 266027, check in 27/7/2026, check out 29/7/2026, cho 2 người lớn, có 2 trẻ em 0 tuổi và 5 tuổi, giá nhiều nhất là 10 triệu.",
]

for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs)

In [ ]:
test_cases = [
    "Tôi muốn xem phòng ở khách sạn có ID là 3215094, check in 27/7/2026, check out 29/7/2026, cho 2 người lớn, có 2 trẻ em 0 tuổi và 5 tuổi, giá nhiều nhất là 10 triệu.",
]

for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs)

In [ ]:
test_cases = [
    "Tôi muốn xem phòng ở khách sạn có ID là 15002321, check in 22/7/2026, check out 25/7/2026, cho 2 người lớn, có 2 trẻ em 11 tuổi và 9 tuổi.",
]

for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs)

In [ ]:
test_cases = [
    "Tôi muốn xem review khách sạn có ID là 266027.",
]

for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs)

In [ ]:
test_cases = [
    "Tôi muốn xem tiện ích khách sạn có ID là 266027.",
]

for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs)

In [ ]:
test_cases = [
        "Tôi muốn xem chính sách khách sạn có ID là 266027.",
    ]

for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs)